# M9.12 · CLAHE, bilateral ve non-local means

Yerel kontrast ve kenar koruyan gürültü giderme: uyarlamalı histogram eşitleme (AHE), kontrast sınırlama ve döşeme (tile) enterpolasyonu ile CLAHE; uzaysal ve değer (range) çekirdeğinin çarpımı olan bilateral süzgeç; benzer yamaları tüm arama penceresinden toplayan non-local means ve "yöntem gürültüsü". Alıştırma 7, 8, 9'un hepsi: saf Python'da OpenCV'nin kırpma–yeniden dağıtma kuralı ve bilateral; NumPy'da `cv2.createCLAHE` ile piksel piksel aynı CLAHE; PyTorch'ta türevlenebilir bilateral ve σ'ları gradyanla bulma; en sonda ofset döngüsü + kutu süzgeciyle hızlı NLM.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-12-edge-preserving.html](https://m-gonul.github.io/pytorch-lab/m9-12-edge-preserving.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_12_edge_preserving.py
CHECKS = {
 "b": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import no_imports, need, ensure\n\n\ndef ref_clip(h, clip):\n    h = np.array(h)\n    ex = np.maximum(h - clip, 0).sum(); h = np.minimum(h, clip)\n    n = len(h); h = h + ex // n; r = ex % n\n    if r:\n        step = max(n // r, 1); i = np.arange(n)\n        h = h + ((i % step == 0) & (i // step < r))\n    return h.tolist()\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/cv2 kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"cv2\", \"scipy\", \"skimage\"}))\n\n    def clip():\n        ensure(need(ns, \"clip_histogram\")([10, 2, 0, 0, 0, 0, 0, 4], 4) == [5, 3, 1, 1, 1, 1, 0, 4], \"elle örnek\")\n        rng = np.random.default_rng(0)\n        for _ in range(50):\n            h = rng.integers(0, 60, 256).tolist(); c = int(rng.integers(1, 40))\n            got = ns[\"clip_histogram\"](h, c)\n            ensure(got == ref_clip(h, c), \"rastgele histogram\")\n            ensure(sum(got) == sum(h), \"toplam korunmalı\")\n    R.test(\"clip_histogram (OpenCV kuralı)\", clip)\n\n    def lut():\n        tile = data.camera()[:64, :64]\n        h = np.bincount(tile.ravel(), minlength=256).tolist()\n        got = need(ns, \"tile_lut\")(h, 2.0, 4096)\n        want = cv2.createCLAHE(2.0, (8, 8)).apply(data.camera())\n        ensure(len(got) == 256 and all(0 <= v <= 255 for v in got) and got == sorted(got), \"monoton 256'lık LUT\")\n        ensure(got[255] == 255, \"LUT[255] = 255\")\n        raw = ns[\"tile_lut\"](h, 0, 4096)\n        slope = lambda L: max(b - a for a, b in zip(L, L[1:]))\n        ensure(slope(got) <= slope(raw), \"kırpma eğimi artıramaz\")\n        return f\"LUT'un en büyük eğimi: kırpmasız {slope(raw)}, clip 2.0 ile {slope(got)}\"\n    R.test(\"tile_lut\", lut)\n\n    def bil():\n        rng = np.random.default_rng(1)\n        img = np.clip(data.camera()[200:240, 220:260] + rng.normal(0, 15, (40, 40)), 0, 255).astype(np.uint8)\n        for d, sc, ss in [(7, 30, 5), (9, 50, 3)]:\n            got = np.array(need(ns, \"bilateral\")(img.tolist(), d, sc, ss))\n            want = cv2.bilateralFilter(img, d, sc, ss)\n            diff = np.abs(got - want.astype(int))\n            ensure(diff.max() <= 1 and (diff > 0).mean() < 0.01, f\"d = {d}: en büyük fark {diff.max()}, farklı piksel %{(diff > 0).mean() * 100:.1f}\")\n    R.test(\"bilateral ≈ cv2.bilateralFilter (±1, float32 yuvarlama)\", bil)\n",
 "c": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    rows = []\n    for name in (\"camera\", \"moon\", \"coins\", \"page\"):\n        def t(name=name):\n            im = getattr(data, name)()\n            for clip, tiles in [(2.0, (8, 8)), (4.0, (4, 4)), (1.0, (16, 16)), (40.0, (8, 8)), (0.0, (8, 8)), (2.0, (7, 5))]:\n                got = need(ns, \"clahe_cv\")(im, clip, tiles)\n                want = cv2.createCLAHE(clip, tiles).apply(im)\n                d = int((got != want).sum())\n                ensure(d == 0, f\"clip {clip}, tiles {tiles}: {d} piksel farklı\")\n            rows.append([name, f\"{im.shape[1]} × {im.shape[0]}\", \"dolgu var\" if im.shape[0] % 8 or im.shape[1] % 8 else \"tam bölünür\"])\n        R.test(f\"clahe_cv = cv2.createCLAHE ({name})\", t)\n    if rows:\n        R.table(\"6 parametre kombinasyonunda birebir\", [\"görüntü\", \"boyut\", \"8 × 8 döşemede\"], rows)\n",
 "d": "import numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    rng = np.random.default_rng(0)\n    clean_np = data.camera()[128:384, 128:384]\n    noisy_np = np.clip(clean_np + rng.normal(0, 15, clean_np.shape), 0, 255).astype(np.uint8)\n\n    def cv():\n        for d, sc, ss in [(7, 30.0, 5.0), (9, 50.0, 3.0)]:\n            got = need(ns, \"bilateral_t\")(torch.tensor(noisy_np, dtype=torch.float64), d // 2, sc, ss).numpy()\n            want = cv2.bilateralFilter(noisy_np, d, sc, ss)\n            diff = np.abs(np.clip(np.rint(got), 0, 255) - want)\n            ensure(diff.max() <= 1 and (diff > 0).mean() < 0.005, f\"d = {d}: en büyük fark {diff.max()}, %{(diff > 0).mean() * 100:.2f} piksel\")\n    R.test(\"bilateral_t ≈ cv2.bilateralFilter\", cv)\n\n    def grad():\n        x = torch.tensor(noisy_np[:32, :32], dtype=torch.float64)\n        p = torch.tensor([30.0, 2.0], dtype=torch.float64, requires_grad=True)\n        torch.autograd.gradcheck(lambda a: ns[\"bilateral_t\"](x, 2, a[0], a[1]).sum(), (p,))\n    R.test(\"σ'lara göre türev (gradcheck)\", grad)\n\n    def fit():\n        clean = torch.tensor(clean_np, dtype=torch.float32)\n        noisy = torch.tensor(noisy_np, dtype=torch.float32)\n        P = lambda a: 10 * torch.log10(255 ** 2 / ((a - clean) ** 2).mean()).item()\n        grid = max((P(ns[\"bilateral_t\"](noisy, 3, sc, ss)), sc, ss) for sc in (10, 20, 30, 40, 60) for ss in (0.5, 1, 1.5, 2, 3))\n        sc, ss = need(ns, \"fit_bilateral\")(noisy, clean)\n        pl = P(ns[\"bilateral_t\"](noisy, 3, sc, ss))\n        ensure(pl >= grid[0] - 0.1, f\"öğrenilen {pl:.2f} dB, ızgara {grid[0]:.2f} dB\")\n        R.table(\"σ = 15 Gauss gürültüsü (camera 256²), r = 3\", [\"yöntem\", \"σ_r\", \"σ_s\", \"PSNR (dB)\"],\n                [[\"gürültülü\", \"—\", \"—\", f\"{P(noisy):.2f}\"], [\"ızgara arama (25 nokta)\", grid[1], grid[2], f\"{grid[0]:.2f}\"], [\"gradyanla (150 adım)\", f\"{sc:.1f}\", f\"{ss:.2f}\", f\"{pl:.2f}\"]],\n                note=\"σ_r ≈ 2.5 × gürültü σ'sı: kenar sayılacak genlik gürültünün birkaç katı olmalı.\")\n    R.test(\"σ'ları gradyanla bul\", fit)\n",
 "e": "import numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef ref_nlm(img, h, tw, sw, sigma):\n    H, W = img.shape\n    t, s = tw // 2, sw // 2\n    P = np.pad(img, t + s, mode=\"reflect\")\n    out = np.zeros((H, W))\n    for y in range(H):\n        for x in range(W):\n            cy, cx = y + t + s, x + t + s\n            pp = P[cy - t:cy + t + 1, cx - t:cx + t + 1]\n            num = den = 0.0\n            for dy in range(-s, s + 1):\n                for dx in range(-s, s + 1):\n                    q = P[cy + dy - t:cy + dy + t + 1, cx + dx - t:cx + dx + t + 1]\n                    w = np.exp(-max(((pp - q) ** 2).mean() - 2 * sigma ** 2, 0) / h ** 2)\n                    num += w * P[cy + dy, cx + dx]; den += w\n            out[y, x] = num / den\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız ofset döngüsü\", lambda: max_loops(src, 1, func=\"nlm\"))\n    rng = np.random.default_rng(0)\n    small = np.clip(data.camera()[200:224, 220:244] + rng.normal(0, 15, (24, 24)), 0, 255)\n    for h, tw, sw, sg in [(12.0, 3, 7, 0.0), (20.0, 5, 9, 15.0)]:\n        def t(h=h, tw=tw, sw=sw, sg=sg):\n            got = need(ns, \"nlm\")(torch.tensor(small, dtype=torch.float64), h, tw, sw, sg).numpy()\n            close(got, ref_nlm(small, h, tw, sw, sg), atol=1e-8, what=\"naif NLM\")\n        R.test(f\"nlm = naif tanım (h = {h}, yama {tw}, arama {sw}, σ = {sg})\", t)\n    if \"nlm\" not in ns:\n        return\n    clean = data.camera()[128:384, 128:384]\n    noisy = np.clip(clean + rng.normal(0, 15, clean.shape), 0, 255).astype(np.uint8)\n    P = lambda a: cv2.PSNR(clean, np.clip(np.rint(a), 0, 255).astype(np.uint8))\n    mine = {h: P(ns[\"nlm\"](torch.tensor(noisy, dtype=torch.float32), h, 7, 21, 0.0).numpy()) for h in (8.0, 10.0, 12.0)}\n    cvs = {h: P(cv2.fastNlMeansDenoising(noisy, None, h, 7, 21)) for h in (10, 15, 20)}\n    hm, hc = max(mine, key=mine.get), max(cvs, key=cvs.get)\n    ensure(mine[hm] > cvs[hc] - 1.0, f\"senin en iyin {mine[hm]:.2f}, OpenCV'nin en iyisi {cvs[hc]:.2f}\")\n    t1 = bench(lambda: ns[\"nlm\"](torch.tensor(noisy, dtype=torch.float32), hm, 7, 21), repeat=1, number=1)\n    t2 = bench(lambda: cv2.fastNlMeansDenoising(noisy, None, hc, 7, 21), repeat=3)\n    R.table(\"σ = 15, camera 256², yama 7, arama 21\", [\"yöntem\", \"PSNR (dB)\", \"süre\"],\n            [[\"gürültülü\", f\"{P(noisy):.2f}\", \"—\"], [\"Gauss 5 × 5\", f\"{P(cv2.GaussianBlur(noisy, (5, 5), 0)):.2f}\", \"—\"], [\"bilateral (9, 50, 3)\", f\"{P(cv2.bilateralFilter(noisy, 9, 50, 3)):.2f}\", \"—\"],\n             [f\"senin NLM (h = {hm:g})\", f\"{mine[hm]:.2f}\", fmt_time(t1)], [f\"cv2.fastNlMeansDenoising (h = {hc})\", f\"{cvs[hc]:.2f}\", fmt_time(t2)]],\n            note=\"OpenCV tamsayı aritmetiği ve ağırlık tablolarıyla aynı fikri uygular; h'nin ölçeği farklı olduğu için her biri kendi en iyi h'siyle karşılaştırıldı.\")\n",
 "x1": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    def exact():\n        img = data.camera()[100:160, 300:370]\n        got = need(ns, \"ahe_sliding\")(img, 9)\n        P = cv2.copyMakeBorder(img, 4, 4, 4, 4, cv2.BORDER_REFLECT_101)\n        want = np.zeros_like(img)\n        for y in range(img.shape[0]):\n            for x in range(img.shape[1]):\n                want[y, x] = np.rint(255 * (P[y:y + 9, x:x + 9] <= img[y, x]).mean())\n        ensure(np.array_equal(got, want), \"kayan pencere AHE\")\n    R.test(\"ahe_sliding\", exact)\n\n    def gain():\n        rng = np.random.default_rng(0)\n        img = np.full((128, 128), 120.0); img[:, 64:] = 180\n        img = np.clip(img + rng.normal(0, 2, img.shape), 0, 255).astype(np.uint8)\n        reg = (40, 90, 10, 50)\n        he = cv2.equalizeHist(img); ahe = ns[\"ahe_sliding\"](img, 31); cl = cv2.createCLAHE(2.0, (4, 4)).apply(img)\n        g = {k: need(ns, \"noise_gain\")(img, v, reg) for k, v in ((\"HE\", he), (\"AHE\", ahe), (\"CLAHE\", cl))}\n        ensure(g[\"AHE\"] > g[\"CLAHE\"] and g[\"AHE\"] > 5, f\"kazançlar {g}\")\n        R.table(\"Düz bölge, giriş gürültüsü σ ≈ 2\", [\"yöntem\", \"gürültü kazancı\"], [[k, f\"{v:.1f}×\"] for k, v in g.items()],\n                note=\"AHE düz bölgenin dar histogramını tüm aralığa yayar: gürültü onlarca kat büyür. CLAHE eğimi sınırlar.\")\n    R.test(\"gürültü büyütme: AHE ≫ CLAHE\", gain)\n",
 "x2": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    rng = np.random.default_rng(0)\n    img = np.clip(data.astronaut()[100:228, 150:278] + rng.normal(0, 12, (128, 128, 3)), 0, 255).astype(np.uint8)\n\n    def col():\n        for d, sc, ss in [(7, 30, 5), (9, 60, 3)]:\n            got = need(ns, \"bilateral_color\")(img, d, sc, ss)\n            want = cv2.bilateralFilter(img, d, sc, ss)\n            diff = np.abs(got.astype(int) - want)\n            ensure(diff.max() <= 1 and (diff > 0).mean() < 0.01, f\"d = {d}: en büyük fark {diff.max()}, %{(diff > 0).mean() * 100:.2f}\")\n    R.test(\"bilateral_color ≈ cv2.bilateralFilter (3 kanal)\", col)\n\n    def joint():\n        g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)\n        a = need(ns, \"joint_bilateral\")(g, g, 9, 40, 3)\n        b = cv2.bilateralFilter(g, 9, 40, 3)\n        ensure(np.abs(a.astype(int) - b).max() <= 1, \"rehber = görüntü iken bilateral ile aynı olmalı\")\n        clean = cv2.cvtColor(data.astronaut()[100:228, 150:278], cv2.COLOR_RGB2GRAY)\n        noisy = np.clip(clean + rng.normal(0, 25, clean.shape), 0, 255).astype(np.uint8)\n        j = ns[\"joint_bilateral\"](noisy, clean, 9, 20, 3)\n        s = cv2.bilateralFilter(noisy, 9, 60, 3)\n        pj, ps = cv2.PSNR(clean, j), cv2.PSNR(clean, s)\n        ensure(pj > ps, \"temiz rehberle ortak bilateral daha iyi olmalı\")\n        return f\"σ = 25 gürültü: bilateral {ps:.2f} dB, temiz rehberle ortak bilateral {pj:.2f} dB (flaş/flaşsız fikri)\"\n    R.test(\"joint_bilateral\", joint)\n",
 "x3": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    clean = data.camera()[128:384, 128:384]\n\n    def mn():\n        m = need(ns, \"method_noise\")(clean, lambda x: cv2.GaussianBlur(x, (5, 5), 0))\n        close(m, clean.astype(float) - cv2.GaussianBlur(clean, (5, 5), 0), what=\"Gauss yöntem gürültüsü\")\n        close(ns[\"method_noise\"](clean, lambda x: x), np.zeros(clean.shape), what=\"özdeşlik süzgeci\")\n    R.test(\"method_noise\", mn)\n\n    def cmp():\n        t = need(ns, \"compare\")(clean)\n        ensure(set(t) == {\"gauss5\", \"median5\", \"bilateral\", \"nlm\"}, \"dört süzgeç\")\n        ensure(t[\"nlm\"][1] < t[\"gauss5\"][1] and t[\"bilateral\"][1] < t[\"gauss5\"][1], \"NLM ve bilateral temiz görüntüye Gauss'tan az dokunmalı\")\n        R.table(\"camera 256², σ = 15\", [\"süzgeç\", \"gürültülüde PSNR\", \"yöntem gürültüsü std (temiz girdi)\"], [[k, f\"{v[0]:.2f}\", f\"{v[1]:.2f}\"] for k, v in t.items()],\n                note=\"Yöntem gürültüsü görüntüsünde Gauss'unki tüm kenarları gösterir; NLM'ninki çoğunlukla yapısız beyaz gürültüye benzer.\")\n    R.test(\"karşılaştırma tablosu\", cmp)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_12_edge_preserving as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**AHE:** tek bir küresel dönüşüm yerine her pikseli **kendi komşuluğunun** histogramıyla eşitle. Çıkış, pikselin komşuluğundaki sırasıyla orantılıdır. Sorun: düz bir bölgede histogram dar bir tepedir, CDF dik çıkar ve küçük gürültü büyük kontrasta dönüşür. Kenar pikselleri için komşuluk yansıtmayla genişletilir (sıfır dolgu ya da kopyalama histogramı bozar).

**CL (kontrast sınırlama):** dönüşümün eğimi histogram yüksekliğiyle orantılıdır, o halde histogramı bir **kırpma eşiğinde** kes ve fazlayı bölmelere eşit dağıt (bazı bölmeler yeniden eşiği aşabilir; istenirse işlem tekrarlanır).

**Döşeme + enterpolasyon:** görüntüyü $8\times8$ (16, 32) döşemeye böl, her döşemenin LUT'unu bir kez hesapla. Her piksel en yakın dört döşeme merkezinin LUT'larıyla dönüştürülüp bilineer karıştırılır; kenar şeridinde iki, köşelerde bir LUT. Kayan pencere (SWAHE) daha doğru ama çok daha pahalıdır; döşemeler bağımsız olduğu için CLAHE paralelleşir.

**OpenCV CLAHE ayrıntıları:** görüntü döşeme sayısına bölünmüyorsa sağ ve alttan `BORDER_REFLECT_101` ile **(tiles − boyut mod tiles)** kadar doldurulur (bölünen eksen de!). Kırpma eşiği $\max(\lfloor\text{clip}\cdot N_{tile}/256\rfloor,1)$; fazlalık önce $\lfloor e/256\rfloor$ olarak her bölmeye, kalan $r$ adet $\max(\lfloor256/r\rfloor,1)$ adımla baştan dağıtılır. LUT $=\mathrm{round}(\mathrm{cdf}\cdot255/N_{tile})$. Enterpolasyon `float32`'de: $t_x=x/w_t-0.5$.

**Bilateral:** $BF[I]_p=\frac1{W_p}\sum_qG_{\sigma_s}(\lVert p-q\rVert)\,G_{\sigma_r}(|I_p-I_q|)\,I_q$. Uzaysal çekirdek yalnız mesafeye, değer çekirdeği yalnız yoğunluk farkına bakar; kenarın öbür yanındaki pikseller değer çekirdeğinde sönümlenir. $\sigma_s$: komşuluk ölçeği (ör. köşegenin %2'si), $\sigma_r$: "kenar sayılan en küçük genlik". Kalın özellikler korunur, $\sim2\sigma_s$'den ince özellikler ortalanır.

**NLM:** her piksel için arama penceresindeki tüm $q$'lar; benzerlik tek değer yerine **yamalar** arasında: $d^2(p,q)=\frac1{|P|}\lVert V_p-V_q\rVert^2$, $w=e^{-\max(d^2-2\sigma^2,0)/h^2}$, $\hat f(p)=\sum_qw\,g(q)/\sum_qw$. Uzaklık cezası yok.

**Yöntem gürültüsü:** temiz görüntüye süzgeci uygula, farkı ($f-D(f)$) incele. Mükemmel süzgeç temiz görüntüye dokunmaz; fark ne kadar küçük ve yapısızsa o kadar iyi.

## 1 · Elle

<!-- çözüm -->

Kırpma örneği: fazlalık $6+0=6$, $\lfloor6/8\rfloor=0$ her bölmeye; kalan 6, adım $\max(\lfloor8/6\rfloor,1)=1$: bölme 0–5'e birer → $[5,3,1,1,1,1,0,4]$, toplam yine 16. $\mathrm{cdf}(2)=9$, $9\cdot7/16=3.94\to4$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**CLAHE (OpenCV kuralı)**

- `cl`: 64 × 64 döşeme, clipLimit = 2.0: tamsayı kırpma eşiği $\lfloor2\cdot4096/256\rfloor$
- `b0`: 8 bölmeli $[10,2,0,0,0,0,0,4]$, eşik 4: kırpma + yeniden dağıtma sonrası bölme 0
- `b6`: Aynı, bölme 6
- `lut2`: Aynı, $L=8$: LUT[2] = round(7 · cdf(2) / 16)
- `xa`: Döşeme genişliği 32, $x=20$: sağdaki döşemenin ağırlığı $x/32-0.5-\lfloor\cdot\rfloor$

**Ağırlıklar**

- `bw`: Bilateral: uzaklık 1, $\sigma_s=2$; fark 20, $\sigma_r=10$: $w$
- `nw`: NLM: yama MSE'si 50, $h=10$: $e^{-d^2/h^2}$

In [ ]:
cevaplar = dict(
    cl=None,
    b0=None,
    b6=None,
    lut2=None,
    xa=None,
    bw=None,
    nw=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"cl":[32,0],"b0":[5,0],"b6":[0,0],"lut2":[4,0],"xa":[0.125,0],"bw":[0.119433,0.0001],"nw":[0.606531,0.0001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `clip_histogram(hist, clip)` — OpenCV kuralı: eşik üstünü kes, fazlayı önce $\lfloor e/n\rfloor$ olarak tüm bölmelere, kalan $r$'yi $i=0$'dan başlayıp $\max(\lfloor n/r\rfloor,1)$ adımla birer birer dağıt ($n$ = bölme sayısı). Yeni liste.
- `tile_lut(hist, clip_limit, total)` — `clip = max(int(clip_limit · total / n), 1)` (clip_limit ≤ 0 ise kırpma yok), `clip_histogram`, LUT[$i$] = `round(cdf[i] · (n − 1) / total)`.
- `bilateral(img, d, sigma_color, sigma_space)` — OpenCV tanımı: `radius = d // 2` (d ≤ 0 ise `round(1.5·sigma_space)`), en az 1; yalnız $\sqrt{i^2+j^2}\le$ radius olan ofsetler; `BORDER_REFLECT_101`; $w=e^{-r^2/2\sigma_s^2}e^{-\Delta^2/2\sigma_c^2}$; çıkış `round(Σwv/Σw)`.

In [ ]:
%%writefile stage_b.py
import math


def clip_histogram(hist, clip):
    ...


def tile_lut(hist, clip_limit, total):
    ...


def bilateral(img, d, sigma_color, sigma_space):
    ...


print(clip_histogram([10, 2, 0, 0, 0, 0, 0, 4], 4))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

`clahe_cv(img, clip=2.0, tiles=(8, 8))` — `cv2.createCLAHE(clip, tiles).apply(img)` ile **birebir** (`tiles = (tilesX, tilesY)`), NumPy, **döngü yok**:

1. Dolgu kuralı (giriş metnine bak), döşeme boyutu $t_h\times t_w$, $N=t_ht_w$.
2. Tüm döşemelerin histogramı tek seferde: döşemeleri `reshape/transpose` ile `(T, N)` yap, `np.add.at` ya da ofsetli `bincount`.
3. Kırpma ve yeniden dağıtma vektörel: kalan dağıtımı `(i % step == 0) & (i // step < r)` maskesiyle.
4. LUT: `float32` `cdf · (255/N)`, `np.rint`.
5. Enterpolasyon `float32`'de: $t_x=x\cdot(1/t_w)-0.5$, $x_1=\lfloor t_x\rfloor$, $a=t_x-x_1$, döşeme indisleri $[0,T_x-1]$'e kırpılır; dört LUT bakışı `lut[ty, tx, v]`; sonuç `np.rint`.

In [ ]:
%%writefile stage_c.py
import numpy as np
import cv2


def clahe_cv(img, clip=2.0, tiles=(8, 8)):
    ...


from skimage import data
print(clahe_cv(data.moon()).mean())

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Türevlenebilir bilateral ve parametrelerini öğrenmek.

- `bilateral_t(img, r, sc, ss)` — `img (H, W)` float tensör (0–255), `sc`, `ss` tensör ya da sayı. `F.pad(…, mode="reflect")` (= `REFLECT_101`), `F.unfold` ile $(2r+1)^2$ komşu; dairesel maske $i^2+j^2\le r^2$; $w=e^{-(i^2+j^2)/2\sigma_s^2}e^{-(I_q-I_p)^2/2\sigma_r^2}$; $\sum wI_q/\sum w$.
- `fit_bilateral(noisy, clean, r=3, steps=150, lr=0.05)` — $\log\sigma_r$ ve $\log\sigma_s$'yi $(\log10,\log1)$'den başlatıp MSE ile Adam; `(sc, ss)` (float) döndür.

Kontrol, `bilateral_t`'nin yuvarlanmış çıkışını `cv2.bilateralFilter` ile karşılaştırır (`d = 7, 9`: OpenCV'nin bazı sürümleri `d ≤ 5`'i kırpan farklı bir koda yönlendirir) ve öğrenilen σ'ların ızgara aramasının en iyisine ulaştığını gösterir.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn.functional as F


def bilateral_t(img, r, sc, ss):
    ...


def fit_bilateral(noisy, clean, r=3, steps=150, lr=0.05):
    ...


x = torch.rand(16, 16) * 255
print(bilateral_t(x, 2, 30.0, 2.0).shape)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`nlm(img, h, tw=7, sw=21, sigma=0.0)` — `img (H, W)` float tensör. Hızlı NLM (ofset döngüsü):

1. `P = F.pad(img, t + s, mode="reflect")`, $t=tw/2$, $s=sw/2$.
2. Her arama ofseti $(d_y,d_x)\in[-s,s]^2$ için (tek döngü, ofset listesi üzerinde): kaydırılmış görüntüyle kare fark haritası; **kutu süzgeci** (`F.avg_pool2d(…, tw, stride=1)`) ile her pikselin yama MSE'si $d^2$ — yama boyutundan bağımsız maliyet.
3. $w=e^{-\max(d^2-2\sigma^2,0)/h^2}$; pay ve paydayı biriktir.

Karmaşıklık: naif $O(HW\cdot sw^2\cdot tw^2)$, bu yöntem $O(HW\cdot sw^2)$.

In [ ]:
%%writefile stage_e.py
import itertools
import torch
import torch.nn.functional as F


def nlm(img, h, tw=7, sw=21, sigma=0.0):
    ...


print(nlm(torch.rand(24, 24) * 255, 10.0, 3, 5).shape)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · HE, kayan pencereli AHE ve CLAHE: gürültü büyütme

- `ahe_sliding(img, w)` — kayan pencereli AHE (SWAHE): her piksel için $w\times w$ komşuluğunda (`BORDER_REFLECT_101`) değeri $\le v$ olan piksel oranı $c$; çıkış `round(255·c)`. `sliding_window_view` ile, döngüsüz.
- `noise_gain(img, out, region)` — `region = (y0, y1, x0, x1)` düz bölgede çıkış std / giriş std.

Kontrol, düz ama hafif gürültülü bir bölgede HE, AHE ve CLAHE'nin gürültüyü kaç kat büyüttüğünü karşılaştırır.

In [ ]:
%%writefile extra_x1.py
import numpy as np
import cv2


def ahe_sliding(img, w):
    ...


def noise_gain(img, out, region):
    ...


from skimage import data
print(ahe_sliding(data.camera()[:64, :64], 15).mean())

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Renkli ve ortak (joint) bilateral

- `bilateral_color(img, d, sc, ss)` — 3 kanallı `uint8`, `cv2.bilateralFilter` ile (`d ≥ 7`) en fazla 1 fark: değer ağırlığı **kanal farklarının mutlak toplamıyla** $e^{-(\sum_c|\Delta_c|)^2/2\sigma_c^2}$, tüm kanallara aynı ağırlık.
- `joint_bilateral(img, guide, d, sc, ss)` — değer ağırlığı `guide`'dan (gri), ortalanan `img`. `guide = img` ise bilateral ile aynı. (Flaşlı/flaşsız fotoğraf birleştirme fikri.)

In [ ]:
%%writefile extra_x2.py
import math
import numpy as np
import cv2


def bilateral_color(img, d, sc, ss):
    ...


def joint_bilateral(img, guide, d, sc, ss):
    ...


from skimage import data
print(bilateral_color(data.astronaut()[:64, :64], 7, 30, 5).shape)

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Yöntem gürültüsü: hangi süzgeç görüntüye en az dokunur?

- `method_noise(clean, fn)` — `clean.astype(float) − fn(clean)`.
- `compare(clean, sigma=15, seed=0)` — sözlük: `{"gauss5", "median5", "bilateral", "nlm"}` her biri için `(psnr_on_noisy, method_noise_std)`; süzgeçler: `cv2.GaussianBlur(5×5, σ=0)`, `cv2.medianBlur(5)`, `cv2.bilateralFilter(9, 50, 3)`, `cv2.fastNlMeansDenoising(h=15, 7, 21)`.

In [ ]:
%%writefile extra_x3.py
import numpy as np
import cv2


def method_noise(clean, fn):
    ...


def compare(clean, sigma=15, seed=0):
    ...


from skimage import data
print(compare(data.camera()[:128, :128]))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: HDRNet: derin bilateral öğrenme

[Gharbi, Chen, Barron, Hasinoff, Durand — Deep Bilateral Learning for Real-Time Image Enhancement (SIGGRAPH 2017)](https://arxiv.org/abs/1707.02880)

Düşük çözünürlüklü bir CNN'in bilateral ızgara (bilateral grid) içinde yerel afin renk dönüşümleri tahmin ettiği, tam çözünürlüğe ise öğrenilen bir rehber (guide) haritasıyla "dilimleme" (slicing) yaparak geçtiği HDRNet'i uygula. MIT-Adobe FiveK'den bir alt kümeyle (ya da kendi ürettiğin bir hedefle: CLAHE + doygunluk artırma) eğit.


**Adımlar**

1. Bilateral ızgara: (x, y, yoğunluk) 3-B ızgara; trilineer dilimleme işlemini `grid_sample` (5-B) ile türevlenebilir yaz.
2. Düşük çözünürlük yolu: yerel ve küresel öznitelikler, ızgara başına 3 × 4 afin katsayı.
3. Tam çözünürlük yolu: piksel başına öğrenilen rehber $g(p)$; çıkış $A[x,y,g(p)]\,[r,g,b,1]^T$.
4. Hedef olarak bu modüldeki CLAHE'yi (Lab'ın L kanalında) kullanıp ağın onu taklit etmesini dene; 4K görüntüde süre ölç.

**Kabul ölçütleri**

- Test PSNR ≥ 25 dB (kendi hedefin için) ve 1080p'de CPU'da bile < 100 ms çıkarım.
- Rehber haritasının görselleştirmesi.
- Izgara derinliği (8, 16) ve uzaysal çözünürlük (8, 16) ablasyonu.

## Kaynaklar

- [Zuiderveld — Contrast Limited Adaptive Histogram Equalization (Graphics Gems IV, 1994)](https://doi.org/10.1016/B978-0-12-336156-1.50061-6) — ©
- [Paris, Kornprobst, Tumblin, Durand — A Gentle Introduction to Bilateral Filtering and its Applications (SIGGRAPH kursu)](https://people.csail.mit.edu/sparis/bf_course/) — ©
- [Buades, Coll, Morel — A non-local algorithm for image denoising (CVPR 2005)](https://doi.org/10.1109/CVPR.2005.38) — ©
- [Buades, Coll, Morel — Non-Local Means Denoising (IPOL 2011)](https://doi.org/10.5201/ipol.2011.bcm_nlm) — CC-BY-NC-SA
- [OpenCV — imgproc/src/clahe.cpp, bilateral_filter](https://github.com/opencv/opencv/blob/4.x/modules/imgproc/src/clahe.cpp) — Apache-2.0